In [ ]:
#passive scanner
!mkdir -p ripe_db
!mkdir -p csv
!wget ftp://ftp.afrinic.net/pub/dbase/afrinic.db.gz -O ripe_db/afrinic.db.gz
!wget ftp://ftp.apnic.net/pub/apnic/whois/apnic.db.inetnum.gz -O ripe_db/apnic.db.inetnum.gz
!wget ftp://ftp.apnic.net/pub/apnic/whois/apnic.db.inet6num.gz -O ripe_db/apnic.db.inet6num.gz
!wget ftp://ftp.lacnic.net/pub/stats/lacnic/delegated-lacnic-extended-latest -O ripe_db/delegated-lacnic-extended-latest
!wget ftp://ftp.ripe.net/ripe/dbase/split/ripe.db.inetnum.gz -O ripe_db/ripe.db.inetnum.gz
!wget ftp://ftp.ripe.net/ripe/dbase/split/ripe.db.inet6num.gz -O ripe_db/ripe.db.inet6num.gz
#!wget ftp://ftp.arin.net/pub/rr/arin.db -O ripe_db/arin.db

In [ ]:
target_regex = r'\b ebay ' #company name
#api censys
UID = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
SECRET = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
#api shodan
api_shodan = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

In [ ]:
import gzip
import time
from multiprocessing import cpu_count, Queue, Process, current_process
import logging
import pandas as pd
import re
import os.path
from netaddr import iprange_to_cidrs
import math
import subprocess
import matplotlib as plt
import matplotlib.ticker as mtick
import shodan
import sys
import requests
import json

In [ ]:
###variables
#api shodan
api = shodan.Shodan(api_shodan)
#target and dataframe
targets = []
df = []
#logger
logger = logging.getLogger('perimeter_parser_scanner')
logger.setLevel(logging.DEBUG)
LOG_FORMAT = '%(asctime)s - %(name)s - %(levelname)s - %(processName)s - %(message)s'
formatter = logging.Formatter(LOG_FORMAT)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
stream_handler.setLevel(logging.DEBUG)
logger.addHandler(stream_handler)

In [ ]:
###functions
#filter target
def filter_target(args):
    rule = re.compile(target_regex, flags=re.I | re.X)
    res = rule.findall(args)
    if res:
        #logger.info(res)
        return res
        
#shodan api
def query_parse(net):
    query = 'net:"{}"'.format(net)
    return query

#shodan api - rangelist = inetnum(list),range_cidr_search(["x.x.x.x","x.x.x.y"])
def range_cidr_search(rangelist):
    n = 0
    for cidr in rangelist:
        n+=1
        logger.info("{} - {} Added".format(n, cidr))
        try:
            query = query_parse(cidr)
            result = api.search(query)
            time.sleep(1) #timing
            for service in result["matches"]:
                targets.append(service)
        except Exception as e:
            print("Error: {}".format(e))
            exit()
    logger.info("Finished")
    return targets

#censys api - censys_search({"query","ip: x.x.x.x"})
def censys_search(query):
    api_url = "https://censys.io/api/v1"
    url = "{}/search/ipv4".format(api_url)
    res = requests.post(url, data=json.dumps(query), auth=(UID, SECRET))
    if res.status_code != 200:
        print("error occurred: {}".format(res.json()["error"]))
        sys.exit(1)
    return res
#censys api - censys_search_netlist(["x.x.x.x","x.x.x.y"])
def censys_search_netlist(query):
    api_url = "https://censys.io/api/v1"
    url = "{}/search/ipv4".format(api_url)
    data = {}
    n = 0
    for ip in query:
        n+=1
        q = {"query":"ip: {}".format(ip)}
        j = censys_search(q)
        logger.info("{} - {} searched".format(n, ip))
        data[ip] = (j.json())
        time.sleep(1)
    logger.info("Finished")
    return data
#censys api - censys_parse(list_cpe_ips)
def censys_parse(list_cpe_ips):
    results = censys_search_netlist(list_cpe_ips)
    c = pd.DataFrame(data=results)
    result_l = {}
    for a in c:
        d = c[a]
        result = d.results[0]
        result_l[result["ip"]] = ""
        for x in result:
            if x == "protocols":
                result_l[result["ip"]] = result["protocols"]
        return result_l

In [ ]:
#exec parser
#parse targets to ripe.db.csv (exec time 400sec)

FILELIST = ['ripe_db/afrinic.db.gz', 'ripe_db/apnic.db.inet6num.gz', 'ripe_db/apnic.db.inetnum.gz', 'ripe_db/delegated-lacnic-extended-latest', 'ripe_db/ripe.db.inet6num.gz'] #arin.db
NUM_WORKERS = 500
COMMIT_COUNT = 10000
NUM_BLOCKS = 0

with open("csv/ripe.db.csv", "w") as f:
    f.write("inetnum;netname;description;country;maintained_by;created;last_modified\n")

def parse_property(block: str, name: str):
    match = re.findall(u'^{0:s}:\s*(.*)$'.format(name), block, re.MULTILINE)
    if match:
        return " ".join(match)
    else:
        return None

def parse_property_inetnum(block: str):
# IPv4
    match = re.findall('^inetnum:[\s]*((?:\d{1,3}\.){3}\d{1,3}[\s]*-[\s]*(?:\d{1,3}\.){3}\d{1,3})', block, re.MULTILINE)
    if match:
        ip_start = re.findall('^inetnum:[\s]*((?:\d{1,3}\.){3}\d{1,3})[\s]*-[\s]*(?:\d{1,3}\.){3}\d{1,3}', block, re.MULTILINE)[0]
        ip_end = re.findall('^inetnum:[\s]*(?:\d{1,3}\.){3}\d{1,3}[\s]*-[\s]*((?:\d{1,3}\.){3}\d{1,3})', block, re.MULTILINE)[0]
        cidrs = iprange_to_cidrs(ip_start, ip_end)
        return '{}'.format(cidrs[0])
# IPv6
    else:
        match = re.findall('^inet6num:[\s]*([0-9a-fA-F:\/]{1,43})', block, re.MULTILINE)
        if match:
            return match[0]
# LACNIC translation for IPv4
        else:
            match = re.findall('^inet4num:[\s]*((?:\d{1,3}\.){3}\d{1,3}/\d{1,2})', block, re.MULTILINE)
            if match:
                return match[0]
            else:
                return None

def read_blocks(filename: str) -> list:
    if filename.endswith('.gz'):
        f = gzip.open(filename, mode='rt', encoding='ISO-8859-1')
    else:
        f = open(filename, mode='rt', encoding='ISO-8859-1')
    single_block = ''
    blocks = []

# Translation for LACNIC DB
    if filename == 'delegated-lacnic-extended-latest':
        for line in f:
            if line.startswith('lacnic'):
                elements = line.split('|')
                if len(elements) >= 7:
                    single_block = ''
                    if elements[2] == 'ipv4':
                        single_block += 'inet4num: ' + elements[3] + '/' + str(int(math.log(4294967296/int(elements[4]),2))) + '\n'
                    elif elements[2] == 'ipv6':
                        single_block += 'inet6num: ' + elements[3] + '/' + elements[4] + '\n'
                    else:
                        continue
                    if len(elements[1]) > 1:
                        single_block += 'country: ' + elements[1] + '\n'
                    if elements[5].isnumeric():
                        single_block += 'last-modified: ' + elements[5] + '\n'
                    single_block += 'descr: ' + elements[6] + '\n'
                    blocks.append(single_block)

# All other DBs goes here
    else:
        for line in f:
            if line.startswith('%') or line.startswith('#') or line.startswith('remarks:') or line.startswith(' '):
                continue
            # block end
            if line.strip() == '':
                if single_block.startswith('inetnum:') or single_block.startswith('inet6num:'):
                    blocks.append(single_block)
                    single_block = ''
                    # comment out to only parse x blocks
                    # if len(blocks) == 100:
                    #    break
                else:
                    single_block = ''
            else:
                single_block += line
    f.close()
    logger.info('Got {} blocks'.format(len(blocks)))
    global NUM_BLOCKS
    NUM_BLOCKS = len(blocks)
    return blocks


def parse_blocks(jobs: Queue):
    counter = 0
    BLOCKS_DONE = 0
    df = pd.DataFrame(columns=['inetnum','netname','description','country','maintained_by','created','last_modified'])
    start_time = time.time()
    with open('csv/ripe.db.csv', 'a', encoding='utf-8') as f:
      while True:
          block = jobs.get()
          if block is None:
              break
          inetnum = parse_property_inetnum(block)
          netname = parse_property(block, 'netname')
          description = parse_property(block, 'descr')
          country = parse_property(block, 'country')
          maintained_by = parse_property(block, 'mnt-by')
          created = parse_property(block, 'created')
          last_modified = parse_property(block, 'last-modified')

          df_str = "{};{};{};{};{};{};{};\n".format(inetnum, netname, description, country, maintained_by, created, last_modified)
          comp = filter_target(df_str)
          if comp:
              #logger.debug(comp)
              f.write(df_str)
          counter += 1
          BLOCKS_DONE += 1
          if counter % COMMIT_COUNT == 0:
              logger.debug('committed {} blocks ({} seconds) {:.1f}% done.'.format(counter, round(time.time() - start_time, 2),BLOCKS_DONE * NUM_WORKERS * 100 / NUM_BLOCKS))
              counter = 0
              start_time = time.time()
    logger.debug('committed last blocks')
    logger.debug('{} finished'.format(current_process().name))

def main():
    overall_start_time = time.time()
    for FILENAME in FILELIST:
        if os.path.exists(FILENAME):
            logger.info('parsing database file: {}'.format(FILENAME))
            start_time = time.time()
            blocks = read_blocks(FILENAME)
            logger.info('database parsing finished: {} seconds'.format(round(time.time() - start_time, 2)))
            logger.info('parsing blocks')
            start_time = time.time()
            jobs = Queue()
            workers = []
            # start workers
            logger.debug('starting {} processes'.format(NUM_WORKERS))
            for w in range(NUM_WORKERS):
                p = Process(target=parse_blocks, args=(jobs,))
                p.start()
                workers.append(p)
            # add tasks
            for b in blocks:
                jobs.put(b)
            for i in range(NUM_WORKERS):
                jobs.put(None)
            # wait to finish
            for p in workers:
                p.join()
            logger.info('block parsing finished: {} seconds'.format(round(time.time() - start_time, 2)))
        else:
            logger.info('File {} not found. Please download the dump'.format(FILENAME))

    logger.info('script finished: {} seconds'.format(round(time.time() - overall_start_time, 2)))


if __name__ == '__main__':
    main()


In [ ]:
#read targets
df = pd.read_csv("csv/ripe.db.csv", delimiter=";", names=['inetnum','netname','description','country','maintained_by','created','last_modified'])

In [ ]:
df = df.iloc[1: , :]
df

In [ ]:
df["inetnum"]

In [ ]:
#graph by country
df.groupby(["country"]).size().plot(kind="bar")

In [ ]:
df["country"].value_counts()

In [ ]:
#(exec shodan api)
#shodan search cidr 
rangelist = df["inetnum"]
t = range_cidr_search(rangelist)
shodan_df = pd.DataFrame(data=t)

In [ ]:
shodan_df.to_csv("csv/shodan_full.csv")

In [ ]:
#vulns
shodan_df[(shodan_df["vulns"].str.len() > 2)][["vulns", "ip_str","port"]]

In [ ]:
#port graph
shodan_df.groupby(["port"]).size().plot(kind="bar")

In [ ]:
#port lookup
#shodan_df[(shodan_df["port"] == 5000) & (shodan_df["ip_str"])]["ip_str"]
#shodan_df[(shodan_df["port"] == 9090) & (shodan_df["ip_str"])]["ip_str"]
pd.set_option('display.max_rows', shodan_df.shape[0]+1)
shodan_df[["ip_str","port"]]

In [ ]:
#cpe ips
shodan_ports = dict(shodan_df[(shodan_df["cpe"].str.len() > 2)][["ip_str", "port"]])
shodan_ports["ip_str"]

In [ ]:
#cpe graph
shodan_df[(shodan_df["cpe"].notnull())].explode("cpe", ignore_index=True).groupby(["cpe"]).size().plot(kind="bar")

In [ ]:
#(exec censys api)

#check list of unique cpe ips (use in censys) 
list_cpe_x = shodan_df[shodan_df["cpe"].notnull()].explode("cpe")
list_cpe_ips = list(pd.unique(list_cpe_x[list_cpe_x["cpe"].str.len() > 2]["ip_str"]))
#list_cpe_ips = list(shodan_df[(shodan_df["cpe"].str.len() > 2)]["ip_str"])
out = censys_search_netlist(list_cpe_ips)
out = dict(out)

In [ ]:
#cpe censys parse ports
cens = pd.DataFrame.from_dict(out, orient="index")
cens_dict = {}
for x in cens["results"]:
    protocols = []
    for y in x:
        for z in y["protocols"]:
            z = z.split("/")[0]
            protocols.append(z)
        cens_dict[y["ip"]] = protocols
censys_ports = pd.DataFrame.from_dict(cens_dict, orient="index")
censys_ports.to_csv("csv/censys_ports.csv")
censys_ports

In [ ]:
#cpe port list shodan
exploded_df = shodan_df[shodan_df["cpe"].notnull()].explode("cpe")
exploded_df[exploded_df["cpe"].str.len() > 2][["cpe","ip_str","port"]]

In [ ]:
#show cve vulns
exploded_vulns = shodan_df[(shodan_df["vulns"].notnull())][["ip_str", "vulns","port"]]
exploded_vulns

In [ ]:
#vulns ips str cve
shodan_df[(shodan_df["vulns"].notnull())]["ip_str"]

In [ ]:
#lookup extensions js
exploded_df = shodan_df[shodan_df["cpe"].notnull()].explode("cpe")
exploded_df[exploded_df["cpe"].str.contains("js")]

In [ ]:
#cpe js
exploded_df = shodan_df[shodan_df["cpe"].notnull()].explode("cpe")
exploded_df[exploded_df["cpe"].str.contains("js")][["cpe","ip_str","port"]]
